# 09 - Retrieval Comparison

## Objective

Compare retrieval quality across two RAG pipelines.

### Pipeline 1

- Recursive Character Chunking
- MiniLM Embeddings
- ChromaDB

### Pipeline 2

- Section-Based Chunking
- BGE-M3 Embeddings
- FAISS

## Evaluation Criteria

- Retrieval Relevance
- Context Quality
- Source Accuracy
- Business Process Preservation

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Load Source Documents

In [2]:
from src.config import DOCS_PATH
from src.loaders.document_loader import DocumentLoader

loader = DocumentLoader(DOCS_PATH)

documents = loader.load()

print(
    f"Source Pages: {len(documents):,}"
)

Starting Document Loading...

Loading PDF : iSupplier_portal.pdf

Loading PDF : Purchasing_manual.pdf


DOCUMENT LOADING SUMMARY
PDF Files Loaded   : 2
DOCX Files Loaded  : 0
XLSX Files Loaded  : 0
Total Documents    : 1502
Source Pages: 1,502


## Rebuild Pipeline 1

Recursive Chunking + MiniLM + ChromaDB

In [3]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

recursive_chunks = (
    recursive_splitter.split_documents(
        documents
    )
)

print(
    f"Recursive Chunks: {len(recursive_chunks):,}"
)

Recursive Chunks: 3,519


## Detect Available Hardware

In [4]:
import torch

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    f"Device: {device}"
)

if torch.cuda.is_available():

    print(
        f"GPU: {torch.cuda.get_device_name(0)}"
    )

Device: cuda
GPU: NVIDIA RTX A5000 Laptop GPU


## Load MiniLM Embedding Model

In [5]:
from langchain_huggingface import (
    HuggingFaceEmbeddings
)

minilm_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={
        "device": device
    }
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Build ChromaDB Retrieval Pipeline

In [6]:
from langchain_chroma import Chroma

chroma_db = Chroma.from_documents(
    documents=recursive_chunks,
    embedding=minilm_model
)

print(
    "ChromaDB Ready"
)

ChromaDB Ready


## Rebuild Pipeline 2

Section-Based Chunking + BGE-M3 + FAISS

In [7]:
from langchain_core.documents import Document

In [8]:
def create_section_chunks(
    documents,
    target_size=2000
):

    section_chunks = []

    current_text = ""

    current_metadata = None

    for doc in documents:

        page_text = doc.page_content.strip()

        if not page_text:
            continue

        if current_metadata is None:
            current_metadata = doc.metadata.copy()

        current_text += "\n\n" + page_text

        if len(current_text) >= target_size:

            section_chunks.append(
                Document(
                    page_content=current_text.strip(),
                    metadata=current_metadata
                )
            )

            current_text = ""
            current_metadata = None

    if current_text:

        section_chunks.append(
            Document(
                page_content=current_text.strip(),
                metadata=current_metadata
            )
        )

    return section_chunks

In [9]:
section_chunks = create_section_chunks(
    documents
)

print(
    f"Section Chunks: {len(section_chunks):,}"
)

Section Chunks: 880


## Load BGE-M3 Embedding Model

In [10]:
bgem3_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={
        "device": device
    }
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## Create Native FAISS Retrieval Pipeline

In [11]:
import faiss
import numpy as np

In [12]:
section_texts = [
    chunk.page_content
    for chunk in section_chunks
]

section_embeddings = (
    bgem3_model.embed_documents(
        section_texts
    )
)

vectors = np.array(
    section_embeddings,
    dtype=np.float32
)

print(
    f"Vector Shape: {vectors.shape}"
)

Vector Shape: (880, 1024)


In [13]:
dimension = vectors.shape[1]

faiss_index = faiss.IndexFlatL2(
    dimension
)

faiss_index.add(vectors)

print(
    f"FAISS Vectors: {faiss_index.ntotal:,}"
)

FAISS Vectors: 880


## Evaluation Questions

Questions used to compare retrieval quality.

In [14]:
evaluation_questions = [

    "How do I create a purchase requisition?",

    "What is an approved supplier list?",

    "How does autocreate work?",

    "What is iSupplier Portal?",

    "How do I create a blanket purchase agreement?"
]

## Compare Retrieval Results

Retrieve the top result from each pipeline for every evaluation question.

In [15]:
import numpy as np

for question in evaluation_questions:

    print("\n" + "=" * 100)

    print(
        f"QUESTION: {question}"
    )

    print("=" * 100)

    chroma_results = (
        chroma_db.similarity_search(
            question,
            k=1
        )
    )

    query_embedding = (
        bgem3_model.embed_query(
            question
        )
    )

    query_vector = np.array(
        [query_embedding],
        dtype=np.float32
    )

    distances, indices = (
        faiss_index.search(
            query_vector,
            k=1
        )
    )

    print("\nPIPELINE 1")
    print("-" * 80)

    print(
        chroma_results[0].metadata
    )

    print(
        chroma_results[0]
        .page_content[:800]
    )

    print("\nPIPELINE 2")
    print("-" * 80)

    idx = indices[0][0]

    print(
        section_chunks[idx].metadata
    )

    print(
        section_chunks[idx]
        .page_content[:800]
    )


QUESTION: How do I create a purchase requisition?

PIPELINE 1
--------------------------------------------------------------------------------
{'file_type': 'pdf', 'page': 231, 'source': 'Purchasing_manual.pdf'}
system. These people can create requisitions for anyone in the company by identifying 
the appropriate requestor directly on the requisition line. You might also want to limit 
some preparers to internal requisitions and others to purchase requisitions. You can 
also print requisitions to obtain signature approvals if some of the approvers do not 
have access to the system. You can then assign an individual to update the 
authorization status for these requisitions.
Purchase Requisitions
Use the Requisitions window to create requisitions. You must choose the requisition 
type (internal or purchase). You can also provide a description, unlimited notes, and 
defaults for requisition lines. For each requisition line, you choose the item you want to 
order along with the quantity 

# Retrieval Comparison Summary

## Pipeline 1

### Architecture

Recursive Character Chunking
→ MiniLM Embeddings
→ ChromaDB

### Strengths

- Smaller chunks
- More focused retrieval
- Higher precision
- Faster indexing

### Best For

- Targeted questions
- Specific Oracle ERP topics
- Fine-grained retrieval

---

## Pipeline 2

### Architecture

Section-Based Chunking
→ BGE-M3 Embeddings
→ Native FAISS

### Strengths

- Larger context windows
- Preserves Oracle workflow continuity
- Richer business process context
- Better procedural coverage

### Best For

- Oracle business processes
- End-to-end workflows
- Procedural guidance

# Notebook Conclusion

## Retrieval Layer Complete

Both retrieval pipelines were successfully validated.

### Pipeline 1

Recursive Chunking
→ MiniLM
→ ChromaDB

### Pipeline 2

Section-Based Chunking
→ BGE-M3
→ Native FAISS

## Outcome

The retrieved contexts generated by both pipelines are now ready for Large Language Model inference.

## Next Step

Proceed to:

### 10_Qwen_Inference.ipynb

Objectives:

- Submit the same question to both retrieval pipelines
- Supply retrieved context to Qwen3:14B
- Compare generated answers
- Evaluate end-to-end RAG quality